## Dataset Generation - Automated Test Case Creation for Agent Evaluation

This tutorial demonstrates automated test case generation for agent evaluation. You'll learn how to generate diverse, high-quality test datasets using the DatasetGenerator API, including saving and loading datasets for reuse across evaluation runs.

### What You'll Learn
- Generate test cases from scratch using topics
- Generate contextual test cases from agent tools and APIs
- Update existing datasets with edge cases and corner scenarios
- Save datasets to JSON files for reuse
- Load datasets from JSON files
- Use auto-rubric generation for evaluators
- Apply topic planning for diverse test coverage

### Tutorial Details

| Information         | Details                                                                       |
|:--------------------|:------------------------------------------------------------------------------|
| Tutorial type       | Intermediate - Automated dataset generation with persistence                  |
| Tutorial components | Multi-agent system, DatasetGenerator, dataset persistence                     |
| Tutorial vertical   | Agent Evaluation                                                              |
| Example complexity  | Medium                                                                        |
| SDK used            | Strands Agents, Strands Evals                                                 |

### Understanding Dataset Generation

Dataset generation automates test case creation for evaluating AI agents. Instead of manually writing test cases, you use AI to generate diverse, comprehensive test scenarios.

#### Why Use Dataset Generation?

| Manual Creation | Automated Generation |
|:----------------|:---------------------|
| Time-consuming | Generate 10-100+ cases in minutes |
| Limited coverage | Diverse topic coverage via topic planning |
| Hard to anticipate edge cases | Automatically includes edge cases |
| Requires domain expertise | Generates evaluation rubrics automatically |

#### When to Use Dataset Generation

Use dataset generation when you need comprehensive coverage across multiple scenarios, rapid prototyping during development, domain-specific testing based on agent tools, regression testing as your agent evolves, or continuous evaluation across different agent versions.

#### Three Generation Strategies

| Strategy | Method | Best For |
|:---------|:-------|:---------|
| From Scratch | `from_scratch_async()` | New agents, broad coverage, exploratory testing |
| From Context | `from_context_async()` | Testing specific tools, API integration scenarios |
| Update Existing | `update_current_dataset_async()` | Adding edge cases, iterative improvement |

#### Dataset Persistence

| Operation | Method | Use Case |
|:----------|:-------|:---------|
| Save | `dataset.to_file('name.json')` | Preserve for reuse, version control |
| Load | `Dataset.from_file('name.json')` | Consistent evaluation, team sharing |

In [ ]:
import os

### Environment Setup

Configure AWS region and model settings for this tutorial.

In [2]:
import boto3

# AWS Configuration
session = boto3.Session()
AWS_REGION = session.region_name or 'us-east-1'
DEFAULT_MODEL = 'us.anthropic.claude-3-7-sonnet-20250219-v1:0'

### Setup and Imports

Import all necessary libraries for agent creation, dataset generation, and evaluation.

In [3]:
# Standard imports
import asyncio
from typing import Dict, List, Any

# Strands imports
from strands import Agent, tool
from strands.multiagent import GraphBuilder

# Strands Evals imports
from strands_evals import Dataset, Case
from strands_evals.generators import DatasetGenerator
from strands_evals.evaluators import OutputEvaluator

# Display utilities
from IPython.display import Markdown, display

### Multi-Agent System with Parallel Execution

We'll create a multi-agent decision-making system with parallel execution. Specialized agents analyze different aspects of a problem and feed into a final decision-maker.

#### Agent Code

The following multi-agent code demonstrates parallel execution with memory branching, adapted from graph agent patterns in strands-samples.

In [4]:
# Multi-agent code adapted from: /strands-samples/01-tutorials/02-multi-agent-systems/03-graph-agent/

# Create specialized agents for parallel analysis
financial_advisor = Agent(
    name="financial_advisor",
    system_prompt="You are a financial advisor focused on cost-benefit analysis, budget implications, and ROI calculations. Provide concise financial assessment.",
    model=DEFAULT_MODEL
)

technical_architect = Agent(
    name="technical_architect",
    system_prompt="You are a technical architect who evaluates feasibility, implementation challenges, and technical risks. Provide concise technical assessment.",
    model=DEFAULT_MODEL
)

market_researcher = Agent(
    name="market_researcher",
    system_prompt="You are a market researcher who analyzes market conditions, user needs, and competitive landscape. Provide concise market assessment.",
    model=DEFAULT_MODEL
)

risk_analyst = Agent(
    name="risk_analyst",
    system_prompt="You are a risk analyst who synthesizes input from finance, technical, and market experts to identify potential risks, mitigation strategies, and provide a final recommendation.",
    model=DEFAULT_MODEL
)

# Build the agent graph with parallel execution
builder = GraphBuilder()

# Add nodes
builder.add_node(financial_advisor, "finance_expert")
builder.add_node(technical_architect, "tech_expert")
builder.add_node(market_researcher, "market_expert")
builder.add_node(risk_analyst, "risk_analyst")

# Add edges - parallel execution pattern
# Finance expert feeds into both tech and market experts
builder.add_edge("finance_expert", "tech_expert")
builder.add_edge("finance_expert", "market_expert")

# Both tech and market experts feed into risk analyst
builder.add_edge("tech_expert", "risk_analyst")
builder.add_edge("market_expert", "risk_analyst")

# Set entry point
builder.set_entry_point("finance_expert")

# Build the graph
decision_graph = builder.build()

Graph without execution limits may run indefinitely if cycles exist


### Test the Multi-Agent System

Before generating datasets, let's verify the multi-agent system works correctly.

In [5]:
# Test the multi-agent system
test_query = "Should we invest $500K in developing an AI-powered customer service chatbot?"
result = decision_graph(test_query)

# Show execution flow
print("\nExecution Order:")
for node in result.execution_order:
    print(f"  - {node.node_id}")

From a financial perspective, this requires careful analysis:

Key considerations:
- Current customer service costs (staff, training, turnover)
- Expected reduction in support tickets/call volume
- Implementation timeline and ongoing maintenance costs
- Potential impact on customer satisfaction and retention

Before proceeding, I recommend:
1. Calculate the fully-loaded cost per customer interaction now
2. Project realistic reduction in human support hours
3. Estimate implementation timeline for positive ROI (likely 12-24 months)
4. Consider phased implementation to validate assumptions with less initial investment

The $500K may be justified if current support costs exceed $250K annually and you can achieve at least 40% automation, but these assumptions need verification with your specific data.## Technical Feasibility Assessment: AI Market Research Assessment: AI Customer Service Chatbot Customer Service Chatbot Investment

## Market Conditions
- The

## Technical Viability AI chatbo

### Strategy 1: Generate Dataset from Scratch

The `from_scratch_async()` method generates test cases from a list of topics. This strategy is ideal when you want to ensure diverse coverage across multiple domains or scenarios.

#### Key Features
- **Topic-based generation**: Specify topics to ensure comprehensive coverage
- **Auto-rubric generation**: Automatically creates evaluation rubrics
- **Difficulty distribution**: Generates easy, medium, and hard test cases
- **Dataset persistence**: Save to JSON for reuse

In [6]:
# Initialize dataset generator
generator = DatasetGenerator(
    input_type=str,
    output_type=str,
    include_expected_output=True,
    model=DEFAULT_MODEL
)

In [7]:
# Generate dataset from scratch with topics
topics = [
    "technology investments",
    "business process automation",
    "market expansion strategies"
]

# Generate dataset
scratch_dataset = await generator.from_scratch_async(
    topics=topics,
    task_description="Multi-agent decision system that provides recommendations on business investments and strategies",
    num_cases=9,
    evaluator=OutputEvaluator
)

#### Save Dataset to JSON

Save the generated dataset to a JSON file for reuse in future evaluation runs.

In [8]:
# Save dataset to JSON file
scratch_dataset.to_file('scratch_dataset.json')

#### Preview Generated Test Cases

In [9]:
# Display sample test cases
for i, case in enumerate(scratch_dataset.cases[:3], 1):
    case_info = f"""
**Case {i}: {case.name}**
**Input**: {case.input}
**Expected Output**: {case.expected_output}
    """
    display(Markdown(case_info))


**Case 1: Simple Technology Investment Recommendation**
**Input**: Our small software company has $100,000 to invest in new technology. We are considering either upgrading our existing server infrastructure or investing in cloud services. We currently spend $2,000 monthly on server maintenance. Cloud services would cost $2,500 monthly but eliminate maintenance costs. Which option should we choose and why?
**Expected Output**: Recommendation to invest in cloud services with clear financial justification showing the ROI over 3-5 years. Analysis should include: initial capital expenditure vs. operational expenditure comparison, total cost of ownership calculation, and acknowledgment of scalability benefits. The recommendation should conclude that while cloud services have a higher monthly cost, the elimination of maintenance costs and improved scalability justify the investment in the long term.
    


**Case 2: Technology Investment - Semiconductor Industry**
**Input**: Our tech investment firm is considering allocating $150M to the semiconductor industry. We're debating between: (1) Investing in established players like TSMC or Intel that have stable revenue but slower growth, (2) Emerging AI chip startups with promising technology but limited market validation, or (3) Specialized semiconductor companies focused on quantum computing. Current market conditions include chip shortages, geopolitical tensions affecting supply chains, and increasing demand from AI applications. Consider 5-year ROI projections, technological risk factors, and alignment with broader market trends.
**Expected Output**: A comprehensive investment recommendation that:
1. Analyzes each investment option with quantitative projections and qualitative risk assessments
2. Evaluates the semiconductor market trends with data-supported insights on supply constraints and demand drivers
3. Considers geopolitical factors affecting manufacturing and supply chains
4. Provides a recommended allocation strategy across the three options with percentage breakdowns
5. Includes contingency recommendations based on potential market shifts
6. Highlights key milestones and metrics to track for re-evaluation of the investment
7. Analyzes the opportunity cost compared to other tech sectors
8. Addresses potential regulatory concerns in different markets
    


**Case 3: Technology Investment Evaluation**
**Input**: Our mid-size software company ($50M annual revenue) is considering investing $5M in either quantum computing research or AI model development. We have an established client base in financial services but are looking to expand our technology offerings. The quantum computing investment would take 3-4 years to potentially yield products, while the AI investment could produce market-ready solutions within 12-18 months. Our technical team has moderate expertise in AI but would need to hire quantum specialists. Market analysis suggests quantum computing may have higher long-term returns but with significantly more uncertainty. What investment strategy would you recommend and why?
**Expected Output**: A comprehensive recommendation that: 
1. Analyzes both investment options (quantum computing vs. AI) with specific pros and cons for each
2. Considers the company's current capabilities, resource requirements, and timeline differences
3. Evaluates market potential and risk factors quantitatively where possible
4. Provides a clear recommendation with primary and secondary options
5. Suggests a phased approach if appropriate (e.g., initial investment followed by milestone-based additional funding)
6. Recommends specific metrics to track for measuring investment success
7. Addresses potential contingencies if market conditions change
    

### Strategy 2: Generate Dataset from Context

The `from_context_async()` method generates test cases based on your agent's specific context, such as tool definitions, APIs, or documentation. This ensures test cases are relevant to your agent's actual capabilities.

#### Key Features
- **Context-aware generation**: Uses agent tools and APIs to create relevant tests
- **Topic planning**: Optionally expand context into diverse topics
- **Tool-specific testing**: Generates tests that exercise specific tools
- **Auto-rubric generation**: Creates rubrics aligned with context

In [10]:
# Define agent context (tools and capabilities)
agent_context = """
Multi-agent decision system with the following capabilities:

Agents:
- Financial Advisor: Analyzes costs, ROI, budget impact, financial risks
- Technical Architect: Evaluates technical feasibility, implementation complexity, architecture risks
- Market Researcher: Assesses market demand, competition, user needs, market timing
- Risk Analyst: Synthesizes all inputs to provide final recommendation

Decision Flow:
1. Financial analysis runs first
2. Technical and market analysis run in parallel
3. Risk analyst synthesizes all perspectives

Output Format:
- Financial assessment
- Technical assessment
- Market assessment
- Final risk analysis and recommendation
"""

# Generate dataset with topic planning for diversity
context_dataset = await generator.from_context_async(
    context=agent_context,
    task_description="Multi-agent system that evaluates business decisions across financial, technical, and market dimensions",
    num_cases=12,
    num_topics=4,  # Topic planning: expand into 4 diverse topics
    evaluator=OutputEvaluator
)

#### Save Context-Based Dataset

In [11]:
# Save context-based dataset to JSON
context_dataset.to_file('context_dataset.json')

#### Preview Context-Based Test Cases

In [12]:
# Display sample test cases
for i, case in enumerate(context_dataset.cases[:3], 1):
    case_info = f"""
**Case {i}: {case.name}**
**Input**: {case.input}
**Expected Output**: {case.expected_output[:200]}...
    """
    display(Markdown(case_info))


**Case 1: New Mobile App Launch for Local Restaurant Delivery Service**
**Input**: A regional restaurant chain is considering launching a mobile app for food delivery. The initial development cost is estimated at $75,000 with ongoing maintenance of $2,000 per month. They expect to process an additional 100 orders per day through the app with an average profit margin of $5 per order. The app would require integration with their existing POS system and would take approximately 3 months to develop. Their target market is urban professionals aged 25-45 in areas where they already have restaurant locations. Two competitors already have similar apps in the market. The restaurant chain wants a comprehensive analysis to decide if they should proceed with the app development.
**Expected Output**: Financial Assessment:
- Initial investment: $75,000
- Monthly maintenance cost: $2,000
- Monthly revenue increase: $15,000 (100 orders × $5 profit × 30 days)
- Projected ROI: 5 months to break even
- ...
    


**Case 2: New Product Launch: Smart Home Automation System**
**Input**: A mid-sized tech company is considering launching a new smart home automation system that integrates with popular voice assistants and includes proprietary sensors for temperature, motion, and security. The system would require a one-time hardware purchase plus optional monthly subscription for premium features.

Development costs are estimated at $2.5M with a 9-month timeline before market readiness. Manufacturing setup would cost an additional $1.2M. Marketing budget is set at $800K for the first year.

The company expects to price the base hardware at $249 with a $9.99 monthly subscription option. Market research shows growing demand for smart home products, but several established competitors already have 60% market share combined.

The company has experience with mobile apps but limited hardware manufacturing experience. They would need to establish new supply chain relationships and QA processes.

Evaluate this potential product launch across financial, technical, and market dimensions, following the sequential decision flow, and provide a comprehensive recommendation on whether to proceed.
**Expected Output**: Financial Assessment:
- Initial investment required: $4.5M ($2.5M development + $1.2M manufacturing + $0.8M marketing)
- Break-even analysis: Approximately 18,000 units needed to recover initial inves...
    


**Case 3: New Product Launch Evaluation - Smart Home Security System**
**Input**: Your multi-agent decision system needs to evaluate the launch of a new smart home security system with the following details:

Product: HomeGuard Pro - An AI-powered home security system with facial recognition, mobile alerts, and integration with smart home devices.

Financial Information:
- Development cost: $3.2 million
- Manufacturing cost per unit: $120
- Proposed retail price: $299
- Marketing budget: $1.5 million for first year
- Expected sales: 50,000 units in year 1, 80,000 in year 2
- Overhead costs: $800,000 per year

Technical Information:
- Core technology uses proprietary AI algorithms for facial recognition
- Requires cloud infrastructure for data processing
- Mobile app development needed for iOS and Android
- Integration capabilities with 5 major smart home platforms
- Estimated development timeline: 8 months
- Current team has experience with similar systems but needs 4 additional developers

Market Information:
- Total smart home security market: $12 billion, growing at 15% annually
- Three major competitors with 65% market share combined
- Main competitor's similar product priced at $349
- Recent consumer survey shows 72% of homeowners are concerned about security
- Market trend shows increased preference for DIY installation systems
- Recent data privacy regulations could impact facial recognition features

Evaluate whether the company should proceed with launching this product, analyzing financial viability, technical feasibility, and market potential. Provide a comprehensive decision recommendation following the established multi-agent decision flow.
**Expected Output**: Financial Assessment:
- Initial ROI analysis: Based on projected sales and costs, the product would break even in approximately 16 months
- First-year profit margin: Approximately 22% after accounting...
    

### Strategy 3: Update Existing Dataset with Edge Cases

The `update_current_dataset_async()` method extends an existing dataset by adding new test cases. This is ideal for iteratively improving test coverage by adding edge cases, corner scenarios, or addressing gaps discovered in production.

#### Key Features
- **Incremental improvement**: Add tests without starting from scratch
- **Edge case coverage**: Focus on corner cases and failure scenarios
- **Dataset continuity**: Preserves existing tests while adding new ones
- **Rubric updates**: Optionally update evaluation rubrics

#### Load Existing Dataset from JSON

First, let's load one of our previously saved datasets to demonstrate the update workflow.

In [13]:
# Load existing dataset from JSON
loaded_dataset = Dataset.from_file('scratch_dataset.json')

In [14]:
# Update dataset with edge cases
edge_case_context = """
Add edge cases and challenging scenarios:
- Conflicting financial and technical assessments
- High-risk, high-reward decisions
- Decisions with missing or incomplete information
- Time-sensitive decisions requiring rapid analysis
- Decisions involving ethical considerations
- Scenarios where experts disagree
"""

# Update dataset by adding edge cases
updated_dataset = await generator.update_current_dataset_async(
    source_dataset=loaded_dataset,
    task_description="Multi-agent decision system handling complex and edge case scenarios",
    num_cases=6,
    context=edge_case_context,
    add_new_cases=True
)

print(f"\nOriginal dataset: {len(loaded_dataset.cases)} cases")
print(f"Updated dataset: {len(updated_dataset.cases)} cases")
print(f"New cases added: {len(updated_dataset.cases) - len(loaded_dataset.cases)}")
print(f"\nUpdated rubric: {updated_dataset.evaluator.rubric}")


Original dataset: 9 cases
Updated dataset: 15 cases
New cases added: 6

Updated rubric: Scoring should evaluate how effectively the multi-agent decision system handles challenging scenarios by assessing: (1) balanced resolution of conflicting financial and technical assessments with clear rationale, (2) justified risk-reward analysis for high-stakes decisions, (3) ability to make sound recommendations despite information gaps, (4) quality of rapid analysis under time constraints, (5) thoughtful consideration of ethical dimensions, and (6) transparent acknowledgment of expert disagreements with reasoned position-taking. Higher scores should reward outputs that demonstrate adaptable reasoning processes, explicitly address uncertainties, provide contingency recommendations, and maintain decision quality even in edge cases.



#### Save Updated Dataset

In [15]:
# Save updated dataset to JSON
print(f"Total test cases: {len(updated_dataset.cases)}")
updated_dataset.to_file('updated_dataset.json')

Total test cases: 15


#### Preview Edge Cases

In [16]:
# Display newly added edge cases (last 3 cases)
new_cases = updated_dataset.cases[-3:]
for i, case in enumerate(new_cases, 1):
    case_info = f"""
**Edge Case {i}: {case.name}**
**Input**: {case.input}
**Expected Output**: {case.expected_output[:200]}...
    """
    display(Markdown(case_info))


**Edge Case 1: Technology Investment with Conflicting Expert Opinions**
**Input**: Our biotech company is facing a critical investment decision with a 2-week deadline. We have $8M available and must choose between two technologies:

Option A: A gene-editing platform that our R&D team strongly supports. The CTO believes this could revolutionize our drug discovery process and cut development time by 40%. Our financial team, however, projects negative cash flow for at least 4 years and questions the $6.5M price tag given our current burn rate. Two competitors have similar technologies at earlier stages.

Option B: An AI-powered drug screening system with immediate integration potential. Our CFO supports this $4.2M investment, projecting profitability within 18 months. However, our head of R&D believes the technology offers minimal advantage over existing systems and could be obsolete within 3 years.

Recent market analysis shows accelerating investment in gene editing, but also increasing regulatory scrutiny. Our board is divided, with technical directors favoring Option A and financial directors favoring Option B. We need a recommendation that considers both perspectives and provides a clear path forward.
**Expected Output**: A balanced recommendation that:

1. Acknowledges the legitimate concerns from both technical and financial perspectives, treating each with appropriate weight
2. Provides quantitative analysis of both...
    


**Edge Case 2: Technology Investment with Conflicting Expert Assessments**
**Input**: Our biotech company ($75M annual revenue) is facing a critical investment decision with conflicting expert opinions. We have $12M available for strategic investment and are considering a novel gene therapy platform that could revolutionize treatments for rare genetic disorders. 

Our financial team projects negative ROI for the first 4 years with break-even only in year 5, and significant market uncertainties. They recommend against the investment, suggesting we license existing technologies instead.

However, our scientific advisory board strongly advocates for the investment, citing a technological breakthrough that could leapfrog competitors and potentially capture 40% market share in an emerging $2B market. They predict first-mover advantage with 3-4 years before competitors can catch up.

The FDA regulatory pathway remains unclear, with the possibility of either accelerated approval (2 years) or standard approval (5+ years). Two of our competitors are pursuing alternative approaches to the same therapeutic targets.

We need a decision within 30 days before our exclusivity agreement with the technology inventors expires. What course of action would you recommend and why?
**Expected Output**: A comprehensive decision framework that reconciles the conflicting expert opinions, including:

1. Balanced assessment of both financial and scientific perspectives with appropriate weighting of each ...
    


**Edge Case 3: Technology Investment with Conflicting Expert Opinions**
**Input**: You've been brought in to resolve a deadlock in our executive committee regarding a critical $20M technology investment decision. Our CTO strongly advocates for investing in a proprietary quantum encryption technology developed by a startup founded by former government researchers. They claim it will provide us with a 3-5 year advantage in data security for our financial services clients. Our CFO opposes the investment, citing independent technical audits suggesting the technology has fundamental flaws and the startup has burned through $40M in funding with minimal commercial validation. Our CSO (Chief Security Officer) believes the technology shows promise but estimates a 60% chance of failure. Market intelligence indicates two major competitors are pursuing similar technology with expected releases in 18-24 months. The startup is giving us just 72 hours to decide, claiming another financial institution is ready to make an exclusive licensing deal. Our board is split on risk tolerance, with half wanting bold innovation and half prioritizing stability. What investment decision would you recommend, and what conditions or structure would you propose to manage the evident risks?
**Expected Output**: A comprehensive recommendation that demonstrates multi-agent reasoning and includes:

1. Expert Opinion Analysis:
   - Critical evaluation of each executive's perspective with weighting based on exper...
    

### Run Evaluation with Generated Dataset

Now let's evaluate our multi-agent system using one of the generated datasets.

In [17]:
# Define agent task function
def agent_task(case: Case) -> str:
    """
    Execute the multi-agent decision system with the given case input.
    """
    result = decision_graph(case.input)
    return str(result)

In [18]:
# Use first 3 cases for demonstration
eval_dataset = Dataset(
    cases=context_dataset.cases[:3],
    evaluator=context_dataset.evaluator
)

report = eval_dataset.run_evaluations(agent_task)

# Mobile App Financial Analysis

## Cost Structure
- Initial investment: $75,000
- Annual maintenance: $24,000 ($2,000/month)
- First year total cost: $99,000

## Revenue Projection
- Daily additional orders: 100
- Profit per order: $5
- Daily additional profit: $500
- Annual additional profit: $182,500 (365 days)

## ROI Analysis
- Payback period: 6.5 months ($99,000 ÷ $15,208/month)
- First year net profit: $83,500 ($182,500 - $99,000)
- 3-year ROI: 486% ([$182,500 × 3] - $75,000 - [$24,000 × 3]) ÷ $75,000

## Recommendations
1. Proceed with development - financial metrics strongly support the investment
2. Verify the 100 orders/day projection - this is the most critical assumption
3. Consider phased launch to test market response before full deployment
4. Analyze competitors' apps to identify differentiating features
5. Develop contingency plan if order volume is 30% below projection

The investment appears financially sound with positive first-year returns and reasonable payback pe

### Evaluation Results

Display the evaluation results using the auto-generated rubric.

In [19]:
# Display evaluation report
report.run_display()

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.87           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                             Test Case Results                                              
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name                                                        ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ New Mobile App Launch for Local Restaurant Delivery Service │ 0.90  │ ✅        │ ...    │ ...   │
├───────┼─────────────────────────────────────────────────────────────┼───────┼───────────┼────────┼───────┤
│ ▶ 1   │ New Product Launch: Smart Home Automation System            │ 0.80  │ ✅        │ ...    │ ...   │
├───────┼─────────────────────────────────────────────────────────────┼───────┼───────────┼────────┼───────┤
│ ▶ 2   │ New Product Launch Evaluation - Smart Home Security System  │ 0.90  │ ✅        │ ...    │ ...   │
└───────┴─────────────────────────────────────────────────────────────┴───────┴───────────┴────────┴───────┘

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.87           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 Test Case Results                                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ index ┃ name                       ┃ score ┃ test_pass ┃ reason                    ┃ input                      ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ▼ 0   │ New Mobile App Launch for  │ 0.90  │ ✅        │ The multi-agent output    │ A regional restaurant      │
│       │ Local Restaurant Delivery  │       │           │ provides an exceptionally │ chain is considering       │
│       │ Service                    │       │           │ comprehensive analysis    │ launching a mobile app for │
│       │                            │       │           │ across all four key       │ food delivery. The initial │
│       │                            │       │           │ dimensions. Financial     │ development cost is        │
│       │                            │       │           │ assessment includes       │ estimated at $75,000 with  │
│       │                            │       │           │ accurate cost             │ ongoing maintenance of     │
│       │                            │       │           │ calculations ($75k        │ $2,000 per month. They     │
│       │                            │       │           │ initial, $24k annual      │ expect to process an       │
│       │                            │       │           │ maintenance), detailed    │ additional 100 orders per  │
│       │                            │       │           │ ROI analysis (6.5-month   │ day through the app with   │
│       │                            │       │           │ payback), and extensive   │ an average profit margin   │
│       │                            │       │           │ financial projections.    │ of $5 per order. The app   │
│       │                            │       │           │ Technical assessment      │ would require integration  │
│       │                            │       │           │ thoroughly covers POS     │ with their existing POS    │
│       │                            │       │           │ integration complexity,   │ system and would take      │
│       │                            │       │           │ technology stack          │ approximately 3 months to  │
│       │                            │       │           │ requirements,             │ develop. Their target      │
│       │                            │       │           │ implementation            │ market is urban            │
│       │                            │       │           │ challenges, and timeline  │ professionals aged 25-45   │
│       │                            │       │           │ risks. Market assessment  │ in areas where they        │
│       │                            │       │           │ provides rich analysis of │ already have restaurant    │
│       │                            │       │           │ market conditions (15-20% │ locations. Two competitors │
│       │                            │       │           │ growth), user preferences │ already have similar apps  │
│       │                            │       │           │ (78% prefer direct        │ in the market. The         │
│       │                            │       │           │ ordering), competitive    │ restaurant chain wants a   │
│       │                            │       │           │ landscape, and strategic  │ comprehensive analysis to  │
│       │                            │       │           │ considerations. The final │ decide if they should      │
│       │                            │       │           │ risk analysis effectively │ proceed with the app       │
│       │                            │       │           │ synthesizes all inputs    │ development.               │
│       │                            │       │           

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

### Dataset Persistence Workflow

Let's demonstrate a complete workflow showing how datasets can be saved and loaded across different evaluation sessions.

In [20]:
# Summary of dataset persistence workflow
workflow_summary = """
## Dataset Persistence Workflow Summary

### Generated Datasets

**1. scratch_dataset.json**
- Strategy: from_scratch_async()
- Topics: technology investments, business automation, market expansion
- Test cases: 9
- Use case: Broad coverage testing

**2. context_dataset.json**
- Strategy: from_context_async()
- Context: Multi-agent capabilities and decision flow
- Test cases: 12 (with topic planning)
- Use case: Context-aware testing

**3. updated_dataset.json**
- Strategy: update_current_dataset_async()
- Source: scratch_dataset.json + edge cases
- Test cases: 15 (original 9 + 6 new)
- Use case: Iterative improvement with edge cases

### Loading Datasets

```python
# Load any saved dataset
dataset = Dataset.from_file('dataset_name.json')

# Run evaluation
report = dataset.run_evaluations(agent_task)
```

### Benefits

- **Consistency**: Use the same test suite across agent versions
- **Collaboration**: Share datasets with team members
- **Version Control**: Track dataset changes over time
- **Regression Testing**: Ensure new changes don't break existing functionality
- **CI/CD Integration**: Automate evaluation in deployment pipelines
"""

display(Markdown(workflow_summary))


## Dataset Persistence Workflow Summary

### Generated Datasets

**1. scratch_dataset.json**
- Strategy: from_scratch_async()
- Topics: technology investments, business automation, market expansion
- Test cases: 9
- Use case: Broad coverage testing

**2. context_dataset.json**
- Strategy: from_context_async()
- Context: Multi-agent capabilities and decision flow
- Test cases: 12 (with topic planning)
- Use case: Context-aware testing

**3. updated_dataset.json**
- Strategy: update_current_dataset_async()
- Source: scratch_dataset.json + edge cases
- Test cases: 15 (original 9 + 6 new)
- Use case: Iterative improvement with edge cases

### Loading Datasets

```python
# Load any saved dataset
dataset = Dataset.from_file('dataset_name.json')

# Run evaluation
report = dataset.run_evaluations(agent_task)
```

### Benefits

- **Consistency**: Use the same test suite across agent versions
- **Collaboration**: Share datasets with team members
- **Version Control**: Track dataset changes over time
- **Regression Testing**: Ensure new changes don't break existing functionality
- **CI/CD Integration**: Automate evaluation in deployment pipelines


## Best Practices for Dataset Generation

### Choosing the Right Strategy

| Strategy | Use When |
|:---------|:---------|
| `from_scratch_async()` | Starting new project, need broad coverage, no detailed context yet |
| `from_context_async()` | Have well-defined tools/APIs, need tests matching actual capabilities |
| `update_current_dataset_async()` | Improving existing dataset, discovered gaps, adding edge cases |

### Key Recommendations

| Area | Recommendation |
|:-----|:---------------|
| Topic Planning | Use `num_topics=3-6` for diverse coverage |
| Persistence | Save datasets when generation takes time or needs reuse |
| Auto-rubrics | Best with default evaluators; use manual rubrics for specific requirements |
| Iteration | Start broad → add context → refine with edge cases → evaluate → iterate |

## Summary

You've successfully learned how to generate and persist evaluation datasets using Strands Evals. You now understand:

- How to generate test cases from scratch using topics with `from_scratch_async()`
- How to generate contextual test cases from agent capabilities with `from_context_async()`
- How to update existing datasets with edge cases using `update_current_dataset_async()`
- How to save datasets to JSON files with `dataset.to_file()`
- How to load datasets from JSON files with `Dataset.from_file()`
- How to use auto-rubric generation for evaluators
- How to apply topic planning for diverse test coverage
- Best practices for choosing generation strategies

Dataset generation enables you to create comprehensive, diverse test suites quickly, and dataset persistence ensures you can reuse these tests consistently across evaluation runs. This forms the foundation for robust, continuous agent evaluation workflows.